In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import sqlite3
import os

In [2]:
# Connect to the SQLite database
# user = 'sgilson'
#db_path = fr'C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db'
db_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db"
conn = sqlite3.connect(db_path)

In [3]:
#subbasins_shapefile = f'C:/Users/{user}/Research Triangle Institute/IKI Peru Project - Interno/AI2b_Modelacion/Grupos_Modelacion/GIS_WaterALLOC_General/Peru_AHD_with_districts.shp'
subbasins_shapefile = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\GIS_WaterALLOC_General\Peru_AHD_with_districts.shp"
subbasins_gdf = gpd.read_file(subbasins_shapefile).set_index("COMID").to_crs("EPSG:32718")

In [4]:
# Subset to Cuenca Nanay
nanay_gdf = subbasins_gdf[subbasins_gdf["Cuenca"] == "Cuenca Nanay"]

# List of COMIDs in Cuenca Nanay
nanay_comids = nanay_gdf.index.tolist()

In [5]:
# function/sql query to get indicator data and weights for each impact chain and scenario
def get_indicator_data(wascn_id, ic_id, user_id, conn):
    """
    Returns a dataframe of indicator values and weights for one Impact Chain + one WaScnID.
    Includes Factor, TextID, WeightValue, Min, Max, and Order.
    """
    query = """
    WITH
    -- map WaScnID -> ScnID
    ScnMap AS (
        SELECT WaScnID, ScnID
        FROM WaScenarios
        WHERE WaScnID = ?
    ),

    -- WaALLOC values
    WaVals AS (
        SELECT IndID, COMID, Value, WaScnID
        FROM IndValues_WaALLOC
        WHERE WaScnID = ?
    ),

    -- Dynamic values for mapped ScnID
    DynVals AS (
        SELECT ivd.IndID, ivd.COMID, ivd.Value, sm.WaScnID
        FROM IndValues_Dyn ivd
        JOIN ScnMap sm ON ivd.ScnID = sm.ScnID
    ),

    -- Static values (applied everywhere)
    StaticVals AS (
        SELECT ivs.IndID, ivs.COMID, ivs.Value, sm.WaScnID
        FROM IndValues_Static ivs
        JOIN ScnMap sm ON 1=1
    ),

    -- Combine all values
    AllVals AS (
        SELECT * FROM WaVals
        UNION ALL
        SELECT * FROM DynVals
        UNION ALL
        SELECT * FROM StaticVals
    )

    SELECT
        av.WaScnID,
        sm.ScnID,
        ? AS UserID,
        ici.IcID,
        ici.IndID,
        av.COMID,
        av.Value,
        iw.Factor,
        iw.TextID,
        iw.WeightValue,
        ind.Min AS IndMin,
        ind.Max AS IndMax,
        ind."Order" AS IndOrder
    FROM ImpactChain_Indicators ici
    JOIN AllVals av ON ici.IndID = av.IndID
    JOIN ScnMap sm ON av.WaScnID = sm.WaScnID
    LEFT JOIN IndicatorWeights iw
        ON ici.IndID = iw.IndID AND iw.UserID = ?
    JOIN Indicators ind
        ON ici.IndID = ind.IndID
    WHERE ici.IcID = ?;
    """

    df = pd.read_sql(query, conn, params=(wascn_id, wascn_id, user_id, user_id, ic_id))
    return df


In [8]:
def min_max_normalize(df):
    """
    Apply min–max scaling using Indicators.Min and Indicators.Max.
    Then apply directionality:
      - ASC  => keep normalized values
      - DESC => invert normalized values (1 - norm)
    Adds a second column 'ValueNorm_Inverse' = 1 - normalized value
    """
    df = df.copy()

    def scale(row):
        if pd.isna(row["Value"]):
            return np.nan

        denom = row["IndMax"] - row["IndMin"]
        if denom == 0:
            return 0.0

        norm = (row["Value"] - row["IndMin"]) / denom
        norm = np.clip(norm, 0, 1)

        # Apply ASC/DESC inversion
        order_val = str(row.get("IndOrder", "ASC")).strip().upper()
        if order_val == "DESC":
            norm = 1 - norm

        return norm

    df["ValueNorm"] = df.apply(scale, axis=1)
    df["ValueNorm_Inverse"] = 1 - df["ValueNorm"]

    return df

In [10]:
IcID = 9
wascn_id = 1
user_id = 1

df = get_indicator_data(
    wascn_id=wascn_id,
    ic_id=IcID,
    user_id=user_id,
    conn=conn
)

In [11]:
df_nanay = df[df["COMID"].isin(nanay_comids)].copy()

In [12]:
df_nanay = min_max_normalize(df_nanay)

In [15]:
# Filter for DESC indicators and non-NaN values
df_desc = df_nanay[
    (df_nanay["IndOrder"].str.upper() == "DESC") &
    (df_nanay["Value"].notna())
].copy()

# Display the dataframe
print(df_desc[[
    "COMID",
    "IndID",
    "IndOrder",
    "Value",
    "ValueNorm",
    "ValueNorm_Inverse"
]])

           COMID  IndID IndOrder  Value  ValueNorm  ValueNorm_Inverse
14019  304273900    404     DESC   0.29    0.78022            0.21978
14020  304274000    404     DESC   0.29    0.78022            0.21978
14021  304281100    404     DESC   0.29    0.78022            0.21978
14022  304281200    404     DESC   0.29    0.78022            0.21978
14023  304306300    404     DESC   0.29    0.78022            0.21978
...          ...    ...      ...    ...        ...                ...
21548  304894400    501     DESC   4.00    0.99802            0.00198
21549  304896300    501     DESC   1.00    1.00000            0.00000
21556  304911500    501     DESC   2.00    0.99934            0.00066
21566  304941400    501     DESC   1.00    1.00000            0.00000
21569  304952000    501     DESC   6.00    0.99670            0.00330

[391 rows x 6 columns]
